## 📑 Core Max Inflows
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/C4BNGQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 
 
previous_month_last_day = spark.sql(f'''SELECT last_day(add_months('{run_date}', -1)) AS previous_month_last_day''').collect()[0]['previous_month_last_day']  
 
DODPlus1referenceDate = spark.sql(f'''
    SELECT cast(CASE
        WHEN date_add(last_day(add_months('{previous_month_last_day}', 3)), -90) < date_add('{previous_month_last_day}', 1)
        THEN '1900-01-01'
        ELSE date_add('{previous_month_last_day}', 1)
    END as string) AS DODPlus1referenceDate
''').collect()[0]['DODPlus1referenceDate']
 
DODPlus2referenceDate = spark.sql(f'''
    SELECT cast(CASE
        WHEN date_add(last_day(add_months('{previous_month_last_day}', 3)), -90) < date_add('{previous_month_last_day}', 2)
        THEN '1900-01-01'
        ELSE date_add('{previous_month_last_day}', 2)
    END as string) AS DODPlus2referenceDate
''').collect()[0]['DODPlus2referenceDate']

 
som_date = spark.sql(f'''
    SELECT case when IsBusinessDay=1 then '{previous_month_last_day}' else PreviousBusinessDay end AS som_date FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE CalendarDate = '{previous_month_last_day}' AND MONTH(PreviousBusinessDay) = MONTH('{previous_month_last_day}')
    AND YEAR(PreviousBusinessDay) = YEAR('{previous_month_last_day}')
''').collect()[0]['som_date']
som_date = som_date.strftime('%Y-%m-%d') 

###### Schema of the table

In [0]:
# Define the schema name
schema_name = f"`{env_prefix}-cdh-delivery`.repoint"

# Create the schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

table_name = 'Core_MaxInflows'

schema = (
"BusinessDate date," 
"Asset string,"
"ExposureNumber string,"
"DODStartDate date,"
"Balance decimal(19,2),"
"Source string,"
"MaxInflowFlag string,"
"DOD_DPDs int,"
"DOD_Bucket int,"
"DOD_MAX_Bucket int,"
"TargetDate date," 
"InflowMonth string," 
"InflowMonthCode string,"
"ServicedFlag string,"
"HFS boolean,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    som_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### DOD_DPD_FCY_aLL

In [0]:
First = spark.sql(
f'''
select
      AC_SRC_STM_CODE
    , AC_CODE
    , TO_DATE(DOD_DPD_CNTER_STRT_DT, 'd/M/yyyy') AS DOD_DPD_CNTER_STRT_DT
    , TO_DATE(DOD_DPD_CNTER_END_DT, 'd/M/yyyy')  AS DOD_DPD_CNTER_END_DT
    , BusinessDate
from `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy
where BusinessDate = '{som_date}'
and IngestionTimeStamp = (select cast(max(IngestionTimeStamp) as timestamp) from `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy where BusinessDate = '{som_date}')
'''
)  
First.createOrReplaceTempView("First")

# Check if the dataset is empty and print a message accordingly
if First.count() == 0:
    print(f"The required external file extdod_dpd_fcy dataset for date {som_date!r} is empty")

In [0]:
Second = spark.sql(
f'''
select
      AC_SRC_STM_CODE
    , AC_CODE
    , TO_DATE(DOD_DPD_CNTER_STRT_DT, 'd/M/yyyy') AS DOD_DPD_CNTER_STRT_DT
    , TO_DATE(DOD_DPD_CNTER_END_DT, 'd/M/yyyy')  AS DOD_DPD_CNTER_END_DT
    , BusinessDate
from `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy
where BusinessDate = '{DODPlus1referenceDate}'
and IngestionTimeStamp = (select max(IngestionTimeStamp) from `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy where BusinessDate = '{DODPlus1referenceDate}')
and TO_DATE(DOD_DPD_CNTER_STRT_DT, 'd/M/yyyy') = '{DODPlus1referenceDate}'
''')  
Second.createOrReplaceTempView("Second")

# Check if the dataset is empty and print a message accordingly
if First.count() == 0:
    print(f"The required external file extdod_dpd_fcy dataset for date {DODPlus1referenceDate!r} is empty")

In [0]:
Third = spark.sql(
f'''
select
      AC_SRC_STM_CODE
    , AC_CODE
    , TO_DATE(DOD_DPD_CNTER_STRT_DT, 'd/M/yyyy') AS DOD_DPD_CNTER_STRT_DT
    , TO_DATE(DOD_DPD_CNTER_END_DT, 'd/M/yyyy')  AS DOD_DPD_CNTER_END_DT
    , BusinessDate
from `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy
where BusinessDate = '{DODPlus2referenceDate}'
and IngestionTimeStamp = (select max(IngestionTimeStamp) from `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy where BusinessDate = '{DODPlus2referenceDate}')
and TO_DATE(DOD_DPD_CNTER_STRT_DT, 'd/M/yyyy') = '{DODPlus2referenceDate}'
'''
)  
Third.createOrReplaceTempView("Third")

# Check if the dataset is empty and print a message accordingly
if First.count() == 0:
    print(f"The required external file extdod_dpd_fcy dataset for date {DODPlus2referenceDate!r} is empty")

##### DOD_DPD_FCY_aLL

In [0]:
DOD_DPD_FCY_aLL = spark.sql(
f'''
SELECT  A.AC_SRC_STM_CODE
	  , A.AC_CODE
	  , A.DOD_DPD_CNTER_STRT_DT
	  , A.DOD_DPD_CNTER_END_DT
	  , CASE WHEN DATEDIFF(DAY,DOD_DPD_CNTER_STRT_DT,LAST_DAY('{previous_month_last_day}'))+1 < 0 THEN 0
			 ELSE DATEDIFF(DAY,DOD_DPD_CNTER_STRT_DT,LAST_DAY('{previous_month_last_day}'))+1
		END AS DPDs_mtd			                     
      , DATEDIFF(DAY,DOD_DPD_CNTER_STRT_DT,LAST_DAY(add_months('{previous_month_last_day}',1)))+1    AS DPD_Max
      , DATEDIFF(DAY,DOD_DPD_CNTER_STRT_DT,LAST_DAY(add_months('{previous_month_last_day}',2)))+1 	 AS `DPD_Max+1`
      , DATEDIFF(DAY,DOD_DPD_CNTER_STRT_DT,LAST_DAY(add_months('{previous_month_last_day}',3)))+1 	 AS `DPD_Max+2`
      , rank() over (PARTITION BY A.AC_CODE order by BusinessDate asc) as rank_dod
FROM (
  Select * from First
  UNION 
  Select * from Second
  UNION 
  Select * from Third
	) A
WHERE 1=1
	  AND A.AC_SRC_STM_CODE IN ('CMS', 'HSL', 'HST', 'LMS', 'SGL', 'SSE')
	  AND A.DOD_DPD_CNTER_END_DT = '9999-12-31'
'''
)   
DOD_DPD_FCY_aLL.createOrReplaceTempView("DOD_DPD_FCY_aLL")

##### DOD_DPD_FCY

In [0]:
DOD_DPD_FCY = spark.sql(
f'''
SELECT  AC_SRC_STM_CODE
              , AC_CODE
              , DOD_DPD_CNTER_STRT_DT
              , DOD_DPD_CNTER_END_DT
              , DPDs_mtd
              , DPD_Max
              , `DPD_Max+1`
              , `DPD_Max+2`
              , CASE WHEN DPDs_mtd=0                          THEN 0
                     WHEN DPDs_mtd>=1 AND DPDs_mtd<=30    THEN 1
                     WHEN DPDs_mtd>=31 AND DPDs_mtd<=60   THEN 2
                     WHEN DPDs_mtd>=61 AND DPDs_mtd<=90   THEN 3
                     WHEN DPDs_mtd>=91 AND DPDs_mtd<=120  THEN 4
                     WHEN DPDs_mtd>=121 AND DPDs_mtd<=150 THEN 5
                     WHEN DPDs_mtd>=151 AND DPDs_mtd<=180 THEN 6
                     WHEN DPDs_mtd>=181 AND DPDs_mtd<=210 THEN 7
                     WHEN DPDs_mtd>=211 AND DPDs_mtd<=240 THEN 8
                     WHEN DPDs_mtd>=241 AND DPDs_mtd<=270 THEN 9
                     WHEN DPDs_mtd>=271 AND DPDs_mtd<=300 THEN 10
                     WHEN DPDs_mtd>=301 AND DPDs_mtd<=330 THEN 11
                     WHEN DPDs_mtd>=331 AND DPDs_mtd<=360 THEN 12
                     WHEN DPDs_mtd>=361                     THEN 13
                     END AS RiskBucketMTD
              , CASE WHEN DPD_Max=0                      THEN 0
                     WHEN DPD_Max>=1   AND DPD_Max<=30   THEN 1
                     WHEN DPD_Max>=31  AND DPD_Max<=60   THEN 2
                     WHEN DPD_Max>=61  AND DPD_Max<=90   THEN 3
                     WHEN DPD_Max>=91  AND DPD_Max<=120  THEN 4
                     WHEN DPD_Max>=121 AND DPD_Max<=150  THEN 5
                     WHEN DPD_Max>=151 AND DPD_Max<=180  THEN 6
                     WHEN DPD_Max>=181 AND DPD_Max<=210  THEN 7
                     WHEN DPD_Max>=211 AND DPD_Max<=240  THEN 8
                     WHEN DPD_Max>=241 AND DPD_Max<=270  THEN 9
                     WHEN DPD_Max>=271 AND DPD_Max<=300  THEN 10
                     WHEN DPD_Max>=301 AND DPD_Max<=330  THEN 11
                     WHEN DPD_Max>=331 AND DPD_Max<=360  THEN 12
                     WHEN DPD_Max>=361                     THEN 13
                     END AS RiskBucketMax
              , CASE WHEN `DPD_Max+1`=0                           THEN 0
                     WHEN `DPD_Max+1`>=1   AND `DPD_Max+1`<=30  THEN 1
                     WHEN `DPD_Max+1`>=31  AND `DPD_Max+1`<=60  THEN 2
                     WHEN `DPD_Max+1`>=61  AND `DPD_Max+1`<=90  THEN 3
                     WHEN `DPD_Max+1`>=91  AND `DPD_Max+1`<=120 THEN 4
                     WHEN `DPD_Max+1`>=121 AND `DPD_Max+1`<=150 THEN 5
                     WHEN `DPD_Max+1`>=151 AND `DPD_Max+1`<=180 THEN 6
                     WHEN `DPD_Max+1`>=181 AND `DPD_Max+1`<=210 THEN 7
                     WHEN `DPD_Max+1`>=211 AND `DPD_Max+1`<=240 THEN 8
                     WHEN `DPD_Max+1`>=241 AND `DPD_Max+1`<=270 THEN 9
                     WHEN `DPD_Max+1`>=271 AND `DPD_Max+1`<=300 THEN 10
                     WHEN `DPD_Max+1`>=301 AND `DPD_Max+1`<=330 THEN 11
                     WHEN `DPD_Max+1`>=331 AND `DPD_Max+1`<=360 THEN 12
                     WHEN `DPD_Max+1`>=361                      THEN 13
                     END AS `RiskBucketMax+1Month`
              , CASE WHEN `DPD_Max+2`=0                           THEN 0
                     WHEN `DPD_Max+2`>=1   AND `DPD_Max+2`<=30  THEN 1
                     WHEN `DPD_Max+2`>=31  AND `DPD_Max+2`<=60  THEN 2
                     WHEN `DPD_Max+2`>=61  AND `DPD_Max+2`<=90  THEN 3
                     WHEN `DPD_Max+2`>=91  AND `DPD_Max+2`<=120 THEN 4
                     WHEN `DPD_Max+2`>=121 AND `DPD_Max+2`<=150 THEN 5
                     WHEN `DPD_Max+2`>=151 AND `DPD_Max+2`<=180 THEN 6
                     WHEN `DPD_Max+2`>=181 AND `DPD_Max+2`<=210 THEN 7
                     WHEN `DPD_Max+2`>=211 AND `DPD_Max+2`<=240 THEN 8
                     WHEN `DPD_Max+2`>=241 AND `DPD_Max+2`<=270 THEN 9
                     WHEN `DPD_Max+2`>=271 AND `DPD_Max+2`<=300 THEN 10
                     WHEN `DPD_Max+2`>=301 AND `DPD_Max+2`<=330 THEN 11
                     WHEN `DPD_Max+2`>=331 AND `DPD_Max+2`<=360 THEN 12
                     WHEN `DPD_Max+2`>=361                      THEN 13
                     END AS `RiskBucketMax+2Month`
             FROM DOD_DPD_FCY_aLL
             where rank_dod=1
'''
)   
DOD_DPD_FCY.createOrReplaceTempView("DOD_DPD_FCY")



##### Union File

In [0]:
CalendarDatesUnion = spark.sql(
f'''
select SnapshotDate, max(IngestionTimestamp) IngestionTimestamp, rank() over (order by SnapshotDate desc) as rank
from `{env_prefix}-cdh`.externalfiles.extunionfile
where SnapshotDate in('{previous_month_last_day}', last_day(add_months('{previous_month_last_day}',-1 )))
group by SnapshotDate
'''
)  
CalendarDatesUnion.createOrReplaceTempView("CalendarDatesUnion")


# Check if the dataset is empty and print a message accordingly
if CalendarDatesUnion.count() == 0:
    print(f"The required external file extunionfile dataset for dates {previous_month_last_day!r} is empty")

In [0]:
FinalUnion = spark.sql(
f'''
select SnapshotDate
     ,   ECAAssetClass
     ,   ExposureKey
     ,   ExposureNumber
     ,   cast(replace(IFRSGrossCarryingAmount,',','') as decimal(19,2)) as IFRSGrossCarryingAmount
     ,   ExposureSourceSystemKey
     ,   CASE WHEN SnapshotDate = last_day(add_months('{previous_month_last_day}',-1 )) and EBAStatus = 'PL' and FPLUnderProbationFlag = 0
                    and d.RiskBucketMTD>=4 then 'NPL'
              when SnapshotDate = last_day(add_months('{previous_month_last_day}',-1 )) and EBAStatus = 'FPL' and FPLUnderProbationFlag = 0
                    and d.RiskBucketMTD>=4 then 'FNPL'
              when SnapshotDate = last_day(add_months('{previous_month_last_day}',-1 )) and EBAStatus = 'FPL' and FPLUnderProbationFlag = 1
                    and d.RiskBucketMTD>=2 then 'FNPL'
              else EBAStatus END AS EBAStatus
     ,   FPLUnderProbationFlag
     ,   HFSFlag
     ,   BucketCategoryWithMateriality
     ,   ExposureSegmentation
     ,   ProductType AS ProductCode
     ,   ProductType
from `{env_prefix}-cdh`.externalfiles.extunionfile u
Left join DOD_DPD_FCY d on d.AC_CODE = u.ExposureKey
WHERE SnapshotDate = (select cast(SnapshotDate as string)
                    from CalendarDatesUnion
                    where Rank =1
                    )            
and IngestionTimestamp  = (select Max(IngestionTimestamp)
                    from CalendarDatesUnion
                    where rank =1)
'''
)  
FinalUnion.createOrReplaceTempView("FinalUnion")

##### Finalstep

In [0]:
Finalstep = spark.sql(
f'''
select
ECAAssetClass
,ExposureNumber
,DOD_DPD_CNTER_STRT_DT as DODStartDate
,cast(replace(IFRSGrossCarryingAmount,',','') as decimal(19,2)) as IFRSGrossCarryingAmount
,ExposureSourceSystemKey as Source
,CASE WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 AND RiskBucketMTD <= 1 AND RiskBucketMax > 1              THEN 'CFPL Max Inflow'
      WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax > 3              THEN 'FPL Max Inflow'
      WHEN EBAStatus='PL' AND RiskBucketMTD <= 3 AND RiskBucketMax > 3                                                                THEN 'PL Max Inflow'
      WHEN EBAStatus='FNPL' AND RiskBucketMTD <= 3 AND RiskBucketMax > 3                                                       THEN 'FNPL Max Inflow'
      WHEN EBAStatus='NPL' AND RiskBucketMTD <= 3 AND RiskBucketMax > 3                                                               THEN 'NPL Max Inflow'
      WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 AND RiskBucketMTD <= 1 AND RiskBucketMax <= 1 AND `RiskBucketMax+1Month` > 1         THEN 'CFPL Max Inflow'
      WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3     THEN 'FPL Max Inflow'
      WHEN EBAStatus='PL' AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3                                                           THEN 'PL Max Inflow'
      WHEN EBAStatus='FNPL' AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3                                                  THEN 'FNPL Max Inflow'
      WHEN EBAStatus='NPL' AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3                                                          THEN 'NPL Max Inflow'
      WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3             THEN 'FPL Max Inflow'
      WHEN EBAStatus='FNPL' AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3 THEN 'FNPL Max Inflow'
      WHEN EBAStatus='PL' AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3       THEN 'PL Max Inflow'
      WHEN EBAStatus='NPL' AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3      THEN 'NPL Max Inflow'
  END AS MaxInflowFlag
, DPDs_mtd           as DOD_DPDs
, RiskBucketMTD as DOD_Bucket
, CASE WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 AND RiskBucketMTD <= 1 AND RiskBucketMax > 1            THEN RiskBucketMax
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax > 3            THEN RiskBucketMax
       WHEN EBAStatus IN ('PL','FNPL', 'NPL') AND RiskBucketMTD <= 3 AND RiskBucketMax > 3                             THEN RiskBucketMax
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 AND RiskBucketMTD <= 1 AND RiskBucketMax <= 1 AND `RiskBucketMax+1Month` > 1 THEN `RiskBucketMax+1Month`
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3    THEN `RiskBucketMax+1Month`
       WHEN EBAStatus IN ('PL','FNPL', 'NPL')  AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3                        THEN `RiskBucketMax+1Month`
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3 THEN `RiskBucketMax+2Month`
       WHEN EBAStatus IN ('PL','FNPL', 'NPL')  AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3                     THEN `RiskBucketMax+2Month`
   END AS DOD_MAX_Bucket
, CASE WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 THEN cast(DATEADD(DAY,30,DOD_DPD_CNTER_STRT_DT) as date)
       ELSE cast(DATEADD(DAY,90,DOD_DPD_CNTER_STRT_DT) as date)
  END AS TargetDate
, CASE WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 AND RiskBucketMTD <= 1 AND RiskBucketMax > 1            THEN date_FORMAT(DATEADD(MONTH,1,'{previous_month_last_day}'),'yyyy-MM')
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax > 3            THEN date_FORMAT(DATEADD(MONTH,1,'{previous_month_last_day}'),'yyyy-MM')
       WHEN EBAStatus IN ('PL','FNPL', 'NPL') AND RiskBucketMTD <= 3 AND RiskBucketMax > 3                             THEN date_FORMAT(DATEADD(MONTH,1,'{previous_month_last_day}'),'yyyy-MM')
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 AND RiskBucketMTD <= 1 AND RiskBucketMax <= 1 AND `RiskBucketMax+1Month` > 1 THEN date_FORMAT(DATEADD(MONTH,2,'{previous_month_last_day}'),'yyyy-MM')
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3    THEN date_FORMAT(DATEADD(MONTH,2,'{previous_month_last_day}'),'yyyy-MM')
       WHEN EBAStatus IN ('PL','FNPL', 'NPL')  AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3                        THEN date_FORMAT(DATEADD(MONTH,2,'{previous_month_last_day}'),'yyyy-MM')
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3 THEN date_FORMAT(DATEADD(MONTH,3,'{previous_month_last_day}'),'yyyy-MM')
       WHEN EBAStatus IN ('PL','FNPL', 'NPL')  AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3      THEN date_FORMAT(DATEADD(MONTH,3,'{previous_month_last_day}'),'yyyy-MM')
   END AS InflowMonth
, CASE WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 AND RiskBucketMTD <= 1 AND RiskBucketMax > 1            THEN 'M0'
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax > 3            THEN 'M0'
       WHEN EBAStatus IN ('PL','FNPL', 'NPL') AND RiskBucketMTD <= 3 AND RiskBucketMax > 3                             THEN 'M0'
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=1 AND RiskBucketMTD <= 1 AND RiskBucketMax <= 1 AND `RiskBucketMax+1Month` > 1 THEN 'M1'
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3    THEN 'M1'
       WHEN EBAStatus IN ('PL','FNPL', 'NPL')  AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` > 3                        THEN 'M1'
       WHEN EBAStatus='FPL' AND FPLUnderProbationFlag=0 AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3 THEN 'M2'
       WHEN EBAStatus IN ('PL','FNPL', 'NPL')  AND RiskBucketMTD <= 3 AND RiskBucketMax <= 3 AND `RiskBucketMax+1Month` <= 3 AND `RiskBucketMax+2Month` > 3      THEN 'M2'
  END AS InflowMonthCode
, CASE WHEN AC_SRC_STM_CODE='HST' OR ExposureSourceSystemKey ='ATO' OR ProductCode IN ('6090','6091') OR ProductType = 500 OR  
                     (ExposureSegmentation='6' AND ProductType IN ('180', '213', '293', '493', '570', '50038', '82', '83')) THEN 'No'
              ELSE 'Yes'
   END AS ServicedFlag
,HFSFlag             as HFS
from FinalUnion u
Inner join DOD_DPD_FCY d on d.AC_CODE = u.ExposureKey
WHERE ExposureSegmentation > 3
AND cast(replace(IFRSGrossCarryingAmount,',','') as decimal(19,2)) > 0
AND (HFSFlag = 0 OR HFSFlag is null)
AND (Case when SnapshotDate=last_day(add_months('{previous_month_last_day}',-1 )) then RiskBucketMTD < 4
          when SnapshotDate = '{previous_month_last_day}' then BucketCategoryWithMateriality < 4 end)
'''
)  
Finalstep.createOrReplaceTempView("Finalstep")

##### MaxInflows

In [0]:
MaxInflows = spark.sql(
f'''
SELECT 
			 '{som_date}' as BusinessDate
             ,      ECAAssetClass as Asset
            ,       ExposureNumber
            ,       DODStartDate
            ,       IFRSGrossCarryingAmount as Balance 
            ,       Source
            ,       MaxInflowFlag
            ,       DOD_DPDs
            ,       DOD_Bucket
            ,       DOD_MAX_Bucket
            ,       TargetDate
            ,       InflowMonth
            ,       InflowMonthCode
            ,       ServicedFlag
            ,       HFS
		FROM Finalstep
'''
)   
MaxInflows.createOrReplaceTempView("MaxInflows")


##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST(A.BusinessDate AS DATE) AS BusinessDate
, CAST(A.Asset AS STRING) AS Asset 
, CAST(A.ExposureNumber AS STRING) AS ExposureNumber 
, CAST(A.DODStartDate AS DATE) AS DODStartDate
, CAST(A.Balance AS DECIMAL(19,2)) AS Balance
, CAST(A.Source AS STRING) AS Source
, CAST(A.MaxInflowFlag AS STRING) AS MaxInflowFlag
, CAST(A.DOD_DPDs AS INT) AS DOD_DPDs
, CAST(A.DOD_Bucket AS INT) AS DOD_Bucket
, CAST(A.DOD_MAX_Bucket AS INT) AS DOD_MAX_Bucket
, CAST(A.TargetDate AS DATE) AS TargetDate
, CAST(A.InflowMonth AS STRING) AS InflowMonth 
, CAST(A.InflowMonthCode AS STRING) AS InflowMonthCode
, CAST(A.ServicedFlag AS STRING) AS ServicedFlag
, CAST(A.HFS AS BOOLEAN) AS HFS
, CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM MaxInflows A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_MaxInflows: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureNumber"]
check_table(delivery_table_path, som_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
MaxInflows = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {som_date!r}
'''
)
MaxInflows.createOrReplaceTempView("MaxInflows")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/MaxInflows/"
file_name = "MaxInflows_" +som_date+ ".csv"
MaxInflows.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)